In [46]:
%matplotlib widget
try:
    import cupy as xp
    flag_gpu = True
    from cupy.typing import ArrayLike
except:
    import numpy as xp
    flag_gpu = False
    from numpy.typing import ArrayLike

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from scipy.optimize import minimize
from copy import copy

from pandas import DataFrame

https://uspas.fnal.gov/materials/11ODU/Lecture6_Transverse_Beam_Optics_1.pdf

In [117]:
class MicroscopeSection:
    def __init__(self, name:str='', elements:ArrayLike=None, print_fancy:bool=True) -> object:
        self.name = name
        self.elements = elements
        self.print_fancy = print_fancy

        self.ndim = 1

        self.length = xp.sum([e.length for e in self.elements])
    
    def __repr__(self) -> str:
        if self.elements is None:
            return ''
        else:
            columns=['name', 'kind', 'length', 'strength', 'calibration']
            reps = [[e.name, e.kind, e.length, e.strength, e.calibration] for e in self.elements]
            
            if  self.print_fancy:
                display(DataFrame(reps, columns=columns))
                return ''
            else:
                return '\n'.join(['\t'.join([f"{key}: {value}, " for key,value in zip(columns,e)])for e in reps])

    def propogate(self, input:ArrayLike=None, z:None|int|ArrayLike=None, output_per_layer=True) -> ArrayLike:
        # TODO: need to figure out how we want to handle z in a section or full scope.
        #       We could have z reference an element or the full section/scope. 
        if input is None:
            input = xp.zeros((self.ndim*2,1))
            input[::2] = 1
        output = [input]
        for e in self.elements:
            output.append(e.propogate(output[-1], z=z))
        if output_per_layer: return xp.asanyarray(output)
        else: return output[-1]

class MicroscopeSection1D(MicroscopeSection):
    def __init__(self, name:str='', elements:ArrayLike=None, print_fancy:bool=True) -> object:
        super().__init__(name=name, elements=elements, print_fancy=print_fancy)

        self.ndim = 1

    

class Element:
    def __init__(self, kind:str='multi', name:str='Unnamed',
                 length:float=0, strength:float=0, calibration:None|float=None,
                 label:bool=False, print_fancy:bool=True) -> object:
        """General microscope element class.
        $$ 
        T = \begin{matrix}
            C & S\\
            C' & S'
            \end{matrix}
        $$
        where
        $$C=cos(\sqrt{Kl}) \therefore C'=-\sqrt{K}sin({\sqrt{Kl}})$$
        $$S=\frac{1}{\sqrt{K}}sin(\sqrt{Kl}) \therefore S'=sin({\sqrt{Kl}})$$

        Parameters
        ----------
        name : str, optional
            Name given to the lens, by default ''
        strength : float, optional
            Defined as the The focusing strength (K) of a thin lens, by default 0
        calibration : float, optional
            Currnet calibration of the lens in units of ???/A, by default None
        label : bool, optional
            If the element should be labeled when plotted, by default False
        print_fancy : bool, optional
            If a fancy table should be used when printed, by default True
        """
        self.kind = kind
        self.name = name
        self.length = length
        self.strength = strength
        self.calibration = calibration
        self.label = label
        self.print_fancy = print_fancy


    # Could perhaps look at if is not None...:
    def __repr__(self) -> str:
        rep = {'name':self.name,
               'kind':self.kind,
               'length':self.length,
               'strength':self.strength,
               'calibration':self.calibration,
               }
        if  self.print_fancy:
            display(DataFrame({key:[value] for key, value in rep.items()}))
            return ''
        else:
            return '\t'.join([f"{key}: {value}, " for key, value in rep.items()])
    def __copy__(self):
        return type(self)(self.name, self.strength,self.calibration, self.label)

    def propogate(self, input:ArrayLike, z:None|int|ArrayLike=None) -> ArrayLike:
        """Propogate the input through the element.
        For non-zero lenthed elements, fractional positions thorugh the lens can be choosen.

        Parameters
        ----------
        input : ArrayLike
            Initial array to transform.
        z : None | int | ArrayLike, optional
            Scaled propogation positions, by default None
            The positions (or created ones) are scaled from 0-1, with 0 being the start of the lens and 1 the total length.
            If None,      a signle tranformation at the length of the element is performed.
            If int,       an array of size z from 0-1 is created.
            If ArrayLike, the input array is used as is.

        Returns
        -------
        ArrayLike
            Matricies during propogation.
        """
        if z is None: lz = self.length
        elif isinstance(z, int): lz = self.length * xp.linspace(0,1,z)
        elif isinstance(z, ArrayLike): lz = self.length * z
        return xp.eye(2)

class Lens1D(Element):
    def __init__(self, name:str='', 
                 strength:float=0, calibration:float=None,
                 label:float=False, print_fancy:float=True) -> object:
        """Infinitly thin lens.
        $$ 
        T = \begin{matrix}
            1 & 0\\
            -1/f & 0
            \end{matrix}
        $$

        Parameters
        ----------
        name : str, optional
            Name given to the lens, by default ''
        strength : float, optional
            Defined as the focal length, by default 0
            Note this in not the focusing strength (K) and is simply f.
            A thin lens is defind as KL=-1/fas L goes to zero.
        calibration : float, optional
            Currnet calibration of the lens in units of ???/A, by default None
        label : bool, optional
            If the element should be labeled when plotted, by default False
        print_fancy : bool, optional
            If a fancy table should be used when printed, by default True
        """
        super().__init__(kind='lens',name=name,length=0, strength=strength, calibration=calibration,label=label, print_fancy=print_fancy)
        
    def propogate(self, input:ArrayLike, z:None|int|ArrayLike=None) -> ArrayLike:
        """Propogate the input through the lens.

        Parameters
        ----------
        input : ArrayLike
            Initial array to transform.
        z : None | int | ArrayLike, optional
            Scaled propogation positions, by default None
            Meaningless for this zero lengthed element and will not be used.
            The input is retained for consistency.

        Returns
        -------
        ArrayLike
            New focused trajectory.
        """
        T = xp.array([[1, 0],
                      [-1/self.strength, 1]
                      ])
        return T@input

class Drift1D(Element):
    def __init__(self, name='', 
                 length=0,
                 label=False, print_fancy=True):
        super().__init__(kind='drift', name=name, length=length, strength=0, label=label, print_fancy=print_fancy)

    def propogate(self, input:ArrayLike, z:None|int|ArrayLike=None) -> ArrayLike:
        """Propogate the input through the element.
        For non-zero lenthed elements, fractional positions thorugh the lens can be choosen.

        Parameters
        ----------
        input : ArrayLike
            Initial array to transform.
        z : None | int | ArrayLike, optional
            Scaled propogation positions, by default None
            The positions (or created ones) are scaled from 0-1, with 0 being the start of the lens and 1 the total length.
            If None,      a signle tranformation at the length of the element is performed.
            If int,       an array of size z from 0-1 is created.
            If ArrayLike, the input array is used as is.

        Returns
        -------
        ArrayLike
            Matricies during propogation.
        """
        if z is None: lz = self.length
        elif isinstance(z, int): lz = self.length * xp.linspace(0,1,z)
        elif isinstance(z, ArrayLike): lz = self.length * z
        T = xp.array([[1, lz],
                      [0, 1]
                      ])
        return T@input

In [175]:
class MicroscopeSection:
    def __init__(self, name:str='', elements:ArrayLike=None, print_fancy:bool=True) -> object:
        self.name = name
        self.elements = elements
        self.print_fancy = print_fancy

        self.ndim = 1

        self.length = xp.sum([e.length for e in self.elements])
    
    def __repr__(self) -> str:
        if self.elements is None:
            return ''
        else:
            columns=['name', 'kind', 'length', 'strength', 'calibration']
            reps = [[e.name, e.kind, e.length, e.strength, e.calibration] for e in self.elements]
            
            if  self.print_fancy:
                display(DataFrame(reps, columns=columns))
                return ''
            else:
                return '\n'.join(['\t'.join([f"{key}: {value}, " for key,value in zip(columns,e)])for e in reps])

    def propogate(self, input:ArrayLike=None, z:None|int|ArrayLike=None, output_per_layer=True) -> ArrayLike:
        # TODO: need to figure out how we want to handle z in a section or full scope.
        #       We could have z reference an element or the full section/scope. 
        if input is None:
            input = xp.zeros((self.ndim*2,1))
            input[::2] = 1
        output = [input]
        for e in self.elements:
            output.append(e.propogate(output[-1], z=z))
        if output_per_layer: return xp.asanyarray(output)
        else: return output[-1]

class MicroscopeSection1D(MicroscopeSection):
    def __init__(self, name:str='', elements:ArrayLike=None, print_fancy:bool=True) -> object:
        super().__init__(name=name, elements=elements, print_fancy=print_fancy)

        self.ndim = 1

    

class Element:
    def __init__(self, kind:str='multi', name:str='Unnamed',
                 length:float=0, strength:float=0, calibration:None|float=None,
                 ndim:int=3,
                 label:bool=False, print_fancy:bool=True) -> object:
        """General microscope element class.
        $$ 
        T = \begin{matrix}
            C & S\\
            C' & S'
            \end{matrix}
        $$
        where
        $$C=cos(\sqrt{Kl}) \therefore C'=-\sqrt{K}sin({\sqrt{Kl}})$$
        $$S=\frac{1}{\sqrt{K}}sin(\sqrt{Kl}) \therefore S'=sin({\sqrt{Kl}})$$

        Parameters
        ----------
        name : str, optional
            Name given to the lens, by default ''
        kind : stry, optional
            Type of element.
        length : int, optional
            Length of the element, by default=0
        strength : float, optional
            Defined as the The focusing strength (K) of a thin lens, by default 0
        calibration : float, optional
            Currnet calibration of the lens in units of ???/A, by default None
        ndim : int, optional
            The dimensionality of the ray system. The first-order lens matrix will have axes with size 2*ndim, which acounts for the derivatives.
            A 1D element without chromatic contributions will have `ndim=1`.
            A 2D element without chromatic contributions will have `ndim=2`.
            A 2D element with chromatic contributions will have `ndim=3`.
        label : bool, optional
            If the element should be labeled when plotted, by default False
        print_fancy : bool, optional
            If a fancy table should be used when printed, by default True
        """
        self.kind = kind
        self.name = name
        self.length = length
        self.strength = strength
        self.calibration = calibration
        self.ndim = ndim
        self.label = label
        self.print_fancy = print_fancy


    # Could perhaps look at if is not None...:
    def __repr__(self) -> str:
        rep = {'name':self.name,
               'kind':self.kind,
               'length':self.length,
               'strength':self.strength,
               'calibration':self.calibration,
               }
        if  self.print_fancy:
            display(DataFrame({key:[value] for key, value in rep.items()}))
            return ''
        else:
            return '\t'.join([f"{key}: {value}, " for key, value in rep.items()])
    def __copy__(self):
        return type(self)(self.name, self.strength,self.calibration, self.label)
    
    def get_scaled_z(self, zs, allow_array=False):
        if zs is None: lzs = self.length
        elif isinstance(zs, float): lzs = self.length * zs
        elif allow_array:
            if isinstance(zs, int): lzs = self.length * xp.linspace(0,1,zs)
            elif isinstance(zs, ArrayLike): lzs = self.length * zs
        else: ValueError(f'Transform recieved an incorrect type. Recieved type {type(zs)}.')
        return lzs

    def propogate(self, input:ArrayLike, z:None|int|ArrayLike=None) -> ArrayLike:
        """Propogate the input through the element.
        For non-zero lenthed elements, fractional positions thorugh the lens can be choosen.

        Parameters
        ----------
        input : ArrayLike
            Initial array to transform.
        z : None | int | ArrayLike, optional
            Scaled propogation positions, by default None
            The positions (or created ones) are scaled from 0-1, with 0 being the start of the lens and 1 the total length.
            If None,      a signle tranformation at the length of the element is performed.
            If int,       an array of size z from 0-1 is created.
            If ArrayLike, the input array is used as is.

        Returns
        -------
        ArrayLike
            Matricies during propogation.
        """
        if z is None: lz = self.length
        elif isinstance(z, int): lz = self.length * xp.linspace(0,1,z)
        elif isinstance(z, ArrayLike): lz = self.length * z
        return xp.eye(2)

class Lens1D(Element):
    def __init__(self, name:str='', 
                 strength:float=0, calibration:float=None,
                 label:float=False, print_fancy:float=True) -> object:
        """Infinitly thin lens.
        $$ 
        T = \begin{matrix}
            1 & 0\\
            -1/f & 0
            \end{matrix}
        $$

        Parameters
        ----------
        name : str, optional
            Name given to the lens, by default ''
        strength : float, optional
            Defined as the focal length, by default 0
            Note this in not the focusing strength (K) and is simply f.
            A thin lens is defind as KL=-1/fas L goes to zero.
        calibration : float, optional
            Currnet calibration of the lens in units of ???/A, by default None
        label : bool, optional
            If the element should be labeled when plotted, by default False
        print_fancy : bool, optional
            If a fancy table should be used when printed, by default True
        """
        super().__init__(kind='lens',name=name,length=0, strength=strength, calibration=calibration,label=label, print_fancy=print_fancy)
    
    def transform(self, input:ArrayLike, zs:None|float) -> ArrayLike:
        """Transform the input through the element.
        For non-zero lenthed elements, fractional positions thorugh the lens can be choosen.

        Parameters
        ----------
        input : ArrayLike
            Initial array to transform.
        z : None | int , optional
            Scaled propogation positions, by default None
            Meaningless for this zero lengthed element and will not be used, but the input is retained for consistency.

        Returns
        -------
        ArrayLike
            Matrix after transformation.
        """
        
        T = xp.array([[1, 0],
                      [-1/self.strength, 1]
                      ])
        
        return T@input
    
    def propogate(self, input:ArrayLike, z:None|int|ArrayLike=None) -> ArrayLike:
        """Propogate the input through the lens.

        Parameters
        ----------
        input : ArrayLike
            Initial array to transform.
        z : None | float | int | ArrayLike, optional
            Scaled propogation positions, by default None
            Meaningless for this zero lengthed element and will not be used, but the input is retained for consistency.

        Returns
        -------
        ArrayLike
            New focused trajectory.
        """
        T = xp.array([[1, 0],
                      [-1/self.strength, 1]
                      ])
        return T@input

class Drift1D(Element):
    def __init__(self, name='', 
                 length=0,
                 label=False, print_fancy=True):
        """General microscope element class.

        Parameters
        ----------
        name : str, optional
            Name given to the lens, by default ''
        length : int, optional
            Length of the element, by default=0
        calibration : float, optional
            Currnet calibration of the lens in units of ???/A, by default None
        label : bool, optional
            If the element should be labeled when plotted, by default False
        print_fancy : bool, optional
            If a fancy table should be used when printed, by default True
        """
        super().__init__(kind='drift', name=name, length=length, strength=0, ndim=1, label=label, print_fancy=print_fancy)

    def transform(self, input:ArrayLike, zs:None|float) -> ArrayLike:
        """Transform the input through the element.
        For non-zero lenthed elements, fractional positions thorugh the lens can be choosen.

        Parameters
        ----------
        input : ArrayLike
            Initial array to transform.
        z : None | int , optional
            Scaled propogation positions, by default None
            The positions (or created ones) are scaled from 0-1, with 0 being the start of the lens and 1 the total length.
            If None,      a signle tranformation at the length of the element is performed.
            If float,     a scaled position.

        Returns
        -------
        ArrayLike
            Matrix after transformation.
        """
        lzs = self.get_scaled_z(zs)
        
        T = xp.array([[1, lzs],
                      [0, 1]
                      ])
        
        return T@input

    def propogate(self, input:ArrayLike, zs:None|float|int|ArrayLike=None) -> ArrayLike:
        """Propogate the input through the element.
        For non-zero lenthed elements, fractional positions thorugh the lens can be choosen.

        Parameters
        ----------
        input : ArrayLike
            Initial array to transform.
        z : None | float | int | ArrayLike, optional
            Scaled propogation positions, by default None
            The positions (or created ones) are scaled from 0-1, with 0 being the start of the lens and 1 the total length.
            If None,      a signle tranformation at the length of the element is performed.
            If float,     a scaled position.
            If int,       an array of size z from 0-1 is created.
            If ArrayLike, the input array is used as is.

        Returns
        -------
        ArrayLike
            Matricies during propogation.
        """
        lzs = self.get_scaled_z(zs, allow_array=True)

        T = xp.asarray([self.transform(input, zs=s) for s in xp.asarray([lzs]).squeeze()])
        return T

In [176]:
input = xp.array([[1],[0]])
drift0 = Drift1D(name='drift0', length=2)
prop = drift0.propogate(input, zs=5).squeeze()
print(prop.shape)
prop

(5, 2)


array([[1., 0.],
       [1., 0.],
       [1., 0.],
       [1., 0.],
       [1., 0.]])

In [129]:
def test1(x):
    z = xp.sin(x)
    return z
def test2(x):
    z = xp.cos(x)
    return z
x = xp.linspace(0,xp.pi,5)
z = xp.array([test1(x), test2(x)])
z.shape

(2, 5)

In [137]:
x = xp.ones((2,2))
x@xp.ones((*x.shape, 10))

array([[[2., 2., 2., 2., 2., 2., 2., 2., 2., 2.],
        [2., 2., 2., 2., 2., 2., 2., 2., 2., 2.]],

       [[2., 2., 2., 2., 2., 2., 2., 2., 2., 2.],
        [2., 2., 2., 2., 2., 2., 2., 2., 2., 2.]]])

In [118]:
drift0 = Drift1D(name='drift0', length=2)
lens1 = Lens1D(name='lens1', strength=1)
drift1 = Drift1D(name='drift1', length=2)
lens2 = Lens1D(name='lens2', strength=2)
drift2 = Drift1D(name='drift2', length=4)

#lens1, lens2
print(drift0.length, drift1.length)

section1 = MicroscopeSection1D(name='Section1', elements=[drift0, lens1, drift1, lens2, drift2])
section1

2 2


,name,kind,length,strength,calibration
0,drift0,drift,2,0,None
1,lens1,lens,0,1,None
2,drift1,drift,2,0,None
3,lens2,lens,0,2,None
4,drift2,drift,4,0,None


In [119]:
prop = section1.propogate().squeeze()
print(prop.shape)
prop

(6, 2)


array([[ 1. ,  0. ],
       [ 1. ,  0. ],
       [ 1. , -1. ],
       [-1. , -1. ],
       [-1. , -0.5],
       [-3. , -0.5]])

In [121]:
drift0.propogate(, z=5)

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 2 dimensions. The detected shape was (2, 2) + inhomogeneous part.